# DOMINANT node embeddings with t-SNE

This notebook loads one DOMINANT model saved by the benchmark, extracts the node representation produced by the final layer of the shared GNN encoder, and projects those representations to two dimensions with t-SNE.

The initial example uses the best-AUC benchmark checkpoint for the `UAE` dataset. Change `DATASET` below to inspect another dataset.

Label convention: `True` means an IoT node and is plotted in yellow; `False` means a control node and is plotted in blue.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.manifold import TSNE
from torch_geometric.nn import GCN
from pygod.detector.dominant import DOMINANTBase

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from dominant import load_data  # noqa: E402

DATASET = "UAE"
CHECKPOINT = ROOT / "artifacts/results/benchmark/saved_models_best_auc" / DATASET / "pygod_dominant/model.pt"
TSNE_SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"dataset={DATASET}; checkpoint={CHECKPOINT}; device={DEVICE}")

In [ ]:
# Load the same graph feature construction used by the DOMINANT experiments.
_, data = load_data(DATASET)
checkpoint = torch.load(CHECKPOINT, map_location="cpu")
hyperparameters = checkpoint["hyperparameters"]

# DOMINANT uses two encoder GNN layers when num_layers=4.
encoder = DOMINANTBase(
    in_dim=data.num_features,
    hid_dim=int(hyperparameters["hid_dim"]),
    num_layers=int(hyperparameters.get("num_layers", 4)),
    dropout=0.0,
    act=F.relu,
    sigmoid_s=False,
    backbone=GCN,
).to(DEVICE)
encoder.load_state_dict(checkpoint["state_dict"])
encoder.eval()

# The shared encoder output is the representation before either decoder
# generates reconstructed attributes/structure and before anomaly scores.
with torch.no_grad():
    node_embeddings = encoder.shared_encoder(
        data.x.to(DEVICE), data.edge_index.to(DEVICE)
    ).cpu().numpy()

labels = data.y.cpu().numpy().astype(bool)
print(f"nodes={len(labels)}; embedding_shape={node_embeddings.shape}")
print(f"IoT nodes (label=True)={labels.sum()}; control nodes (label=False)={(~labels).sum()}")

In [ ]:
# t-SNE is stochastic, so fix its seed for a repeatable first view.
perplexity = min(30, max(2, (len(node_embeddings) - 1) // 3))
embedding_2d = TSNE(
    n_components=2,
    perplexity=perplexity,
    init="pca",
    learning_rate="auto",
    random_state=TSNE_SEED,
).fit_transform(node_embeddings)
print(f"t-SNE shape={embedding_2d.shape}; perplexity={perplexity}")

In [ ]:
# True labels are IoT nodes (yellow); False labels are control nodes (blue).
plt.figure(figsize=(10, 8))
plt.scatter(
    embedding_2d[~labels, 0], embedding_2d[~labels, 1],
    c="#1f77b4", label="Control (label=False)",
    s=18, alpha=0.75, linewidths=0,
)
plt.scatter(
    embedding_2d[labels, 0], embedding_2d[labels, 1],
    c="#FFD700", edgecolors="#8a6d00",
    label="IoT (label=True)", s=28, alpha=0.95, linewidths=0.3,
)
plt.title(f"DOMINANT final GNN embeddings — {DATASET}")
plt.xlabel("t-SNE component 1")
plt.ylabel("t-SNE component 2")
plt.legend(frameon=True)
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## Interpretation note

t-SNE is useful for visual exploration, but distances and cluster sizes in the two-dimensional plot should not be interpreted as quantitative evidence of separability. The plot is based on the learned encoder representation, not on the DOMINANT anomaly score.